# Task 11 — Tham chiếu Open-Weight LLM (`Qwen/Qwen2.5-7B-Instruct`, `phase2-v1`)

Chọn **Runtime → Change runtime type → GPU** (khuyến nghị **`NVIDIA A100`** hoặc **`NVIDIA L4`** cho `BF16`; GPU < 22 GiB tự chọn **`4bit`**), kiểm tra cấu hình ở **Bước 1** và chạy lần lượt từng cell.

### Quy trình chuẩn hóa theo `phase2-v1` & `Task 08`
1. **Virtualenv riêng (`colab-task11-llm-v1`):** Không cài đè package vào kernel Colab; tự phát hiện `BF16`/`4bit`.
2. **Xác minh phiên bản & giấy phép Hugging Face:** Kiểm tra revision SHA và license của `Qwen/Qwen2.5-7B-Instruct` trước khi suy luận, ghi vào `environment.json`.
3. **Smoke `dev-only` (`20` mẫu)** vào `/content/task11_llm_smoke`, kiểm tra không cắt cụt câu hỏi đích (`max_input_length=2048`).
4. **Dev sweep (`dev-only`)** cho `vsmec` và `neuesc` (`zero_shot` + 3 chiến lược few-shot × 2 prompt style `concise`/`rubric`), hỗ trợ `completion` & `SKIP` từng tổ hợp và đồng bộ liên tục lên Drive phẳng.
5. **Eval `test` 1 lần duy nhất** từ `selection_llm_{domain}.json` đã khóa.
6. **Benchmark chi phí dual-mode (`zero_shot` & `few_shot`)** + tổng hợp bảng hiệu năng–chi phí với `ViSoBERT` và `XLM-R` (nếu có trên Drive).

> **Lưu ý khoa học:** Mọi kết quả LLM phải ghi nhận rõ rủi ro dữ liệu công khai (`UIT-VSMEC`, `NEU-ESC`) có thể đã nằm trong dữ liệu tiền huấn luyện của mô hình.

## 1. Tham số, profile `colab-task11-llm-v1` & đường dẫn Drive phẳng

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, time

T0_SETUP = time.perf_counter()
SESSION_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())

REPO_URL = "https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git"
REPO_REF = "phase2_task11-xlmr-llm"  # đổi sang commit SHA 40 ký tự khi nối lại phiên cũ
PRIVATE_REPO = False

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
PROFILE_ID = "colab-task11-llm-v1"
PROFILES = {
    "colab-task11-llm-v1": {
        "python": "3.13.18",
        "torch": "2.14.1+cu130",
        "cuda": "13.0",
        "pins": {
            "transformers": "4.57.1",
            "huggingface-hub": "0.36.2",
            "tokenizers": "0.22.2",
            "numpy": "2.1.3",
            "pandas": "3.1.0",
            "scikit-learn": "1.6.1",
            "scipy": "1.16.3",
            "PyYAML": "6.0.2",
            "sentencepiece": "0.2.2",
            "protobuf": "7.36.2",
            "safetensors": "0.8.0",
            "tqdm": "4.67.3",
            "accelerate": "1.2.1",
        },
        "bitsandbytes_spec": "bitsandbytes==0.45.3",
    },
}
assert PROFILE_ID in PROFILES

MOUNT_DRIVE = True
USE_DRIVE_BACKUP = True
DRIVE_BACKUP_BASE = Path("/content/drive/MyDrive/Xdomain/Phase2/Task11_LLM")
DRIVE_TASK08_RESULTS_DIR = "/content/drive/MyDrive/Xdomain/Phase2/Task08"
DRIVE_TASK11_XLMR_DIR = "/content/drive/MyDrive/Xdomain/Phase2/Task11_XLMR"

assert Path("/content").is_dir(), "Notebook này yêu cầu môi trường Google Colab (/content)."
REPO_DIR = Path("/content/xdomain-task11-llm")
VENV_DIR = Path(f"/content/phase2-task11-llm-{PROFILE_ID}-venv")
SMOKE_DIR = Path("/content/task11_llm_smoke")
OUTPUT_BASE = Path("/content/task11_llm_outputs")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Session: {SESSION_ID} | Profile: {PROFILE_ID} | Model: {MODEL_NAME}")

## 2. Kiểm tra GPU/VRAM, chọn Precision & mount Google Drive

In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU CUDA trên Colab (Runtime -> Change runtime type -> GPU)."
gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)

if vram_gib >= 40.0 and torch.cuda.is_bf16_supported():
    PRECISION, BATCH_SIZE = "bf16", 32
elif vram_gib >= 22.0 and torch.cuda.is_bf16_supported():
    PRECISION, BATCH_SIZE = "bf16", 16
else:
    PRECISION, BATCH_SIZE = "4bit", 8

expected_profile = PROFILES[PROFILE_ID]
assert torch.__version__ == expected_profile["torch"], (
    f"Torch trên Colab ({torch.__version__}) khác profile {PROFILE_ID} ({expected_profile['torch']})."
)
assert torch.version.cuda == expected_profile["cuda"], (
    f"CUDA trên Colab ({torch.version.cuda}) khác profile {PROFILE_ID} ({expected_profile['cuda']})."
)
print(f"GPU: {gpu_name} | VRAM: {vram_gib} GiB | Precision: {PRECISION} | Batch size: {BATCH_SIZE}")
subprocess.run(["nvidia-smi"], check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Fetch repo, khóa `COMMIT`, chụp `frozen_before.json` & khôi phục từ Drive phẳng

In [ ]:
import base64

def sha256_path(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_copy_file(src: Path, dst: Path) -> None:
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + f".{os.getpid()}.tmp")
    shutil.copy2(src, tmp)
    tmp.replace(dst)

def git_command(args, cwd=None) -> str:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "Thiếu Colab Secret GITHUB_TOKEN khi PRIVATE_REPO=True."
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0="Authorization: Basic " + auth,
        )
    result = subprocess.run(
        ["git", *args], cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Git thất bại: kiểm tra URL, quyền đọc hoặc commit SHA đã được push chưa.\n" + result.stderr
        )
    return result.stdout.strip()

def hydrate_llm_from_flat_drive(drive_root: Path, llm_out_dir: Path, commit: str, profile_id: str) -> None:
    drive_env = drive_root / "environment" / "environment.json"
    if drive_env.is_file():
        d_env = json.loads(drive_env.read_text(encoding="utf-8"))
        assert d_env.get("commit") == commit, f"Drive backup thuộc commit khác ({d_env.get('commit')} != {commit})."
        assert d_env.get("profile_id") == profile_id, f"Drive backup thuộc profile khác."

    restored_items = 0
    for domain in ("vsmec", "neuesc"):
        d_dom = drive_root / domain
        l_dom = llm_out_dir / domain
        if not d_dom.is_dir():
            continue
        for comp_file in sorted(d_dom.glob("dev_completion_*.json")):
            comp = json.loads(comp_file.read_text(encoding="utf-8"))
            if comp.get("complete") is not True:
                continue
            arts = comp.get("artifacts", {})
            if all((d_dom / fname).is_file() and sha256_path(d_dom / fname) == dig for fname, dig in arts.items()):
                for fname in arts:
                    atomic_copy_file(d_dom / fname, l_dom / fname)
                atomic_copy_file(comp_file, l_dom / comp_file.name)
                restored_items += 1
        for extra in (f"dev_sweep_{domain}.csv", f"selection_llm_{domain}.json"):
            if (d_dom / extra).is_file() and not (l_dom / extra).is_file():
                atomic_copy_file(d_dom / extra, l_dom / extra)
        for test_dir in sorted(d_dom.glob("test_*")):
            comp_file = test_dir / "completion.json"
            if not comp_file.is_file():
                continue
            comp = json.loads(comp_file.read_text(encoding="utf-8"))
            if comp.get("complete") is not True:
                continue
            arts = comp.get("artifacts", {})
            if all((test_dir / fname).is_file() and sha256_path(test_dir / fname) == dig for fname, dig in arts.items()):
                for fname in arts:
                    atomic_copy_file(test_dir / fname, l_dom / test_dir.name / fname)
                atomic_copy_file(comp_file, l_dom / test_dir.name / "completion.json")
                restored_items += 1
    if restored_items:
        print(f"Hydrated {restored_items} verified LLM sweep/test artifacts from flat Drive.")

if not REPO_DIR.exists():
    git_command(["clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
assert (REPO_DIR / ".git").is_dir()
if git_command(["diff", "--name-only"], REPO_DIR) or git_command(["diff", "--cached", "--name-only"], REPO_DIR):
    raise RuntimeError("Checkout có thay đổi cục bộ; hãy dùng REPO_DIR sạch.")

git_command(["fetch", "--depth=1", "origin", REPO_REF], REPO_DIR)
git_command(["checkout", "--detach", "FETCH_HEAD"], REPO_DIR)
COMMIT = git_command(["rev-parse", "HEAD"], REPO_DIR)
assert len(COMMIT) == 40

OUTPUT_DIR = OUTPUT_BASE / COMMIT[:12] / PROFILE_ID
RESULTS_DIR = OUTPUT_DIR / "results" / "phase2"
LLM_OUT_DIR = RESULTS_DIR / "llm"
ENV_DIR = OUTPUT_DIR / "environment"
LOG_DIR = OUTPUT_DIR / "logs"
for folder in (RESULTS_DIR, LLM_OUT_DIR, ENV_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DRIVE_SESSION_ROOT = DRIVE_BACKUP_BASE if (MOUNT_DRIVE and USE_DRIVE_BACKUP) else None
if DRIVE_SESSION_ROOT is not None:
    DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)

for rel in (
    "configs/phase2.yaml", "configs/task11_llm.yaml",
    "src/phase2/llm.py", "src/phase2/task11.py", "src/phase2/protocol.py", "src/phase2/run.py",
    "data/clean/neuesc_100.csv", "data/clean/vsmec_train.csv", "data/clean/vsmec_dev.csv", "data/clean/vsmec_test.csv",
    "data/subsets_phase2/MANIFEST.json",
    "data/subsets_phase2/neuesc_f10_s0_ids.csv", "data/subsets_phase2/vsmec_f10_s0_ids.csv",
):
    assert (REPO_DIR / rel).is_file(), f"Thiếu file bắt buộc: {rel}"

frozen_paths = [
    p
    for folder in ("data/clean", "data/splits", "data/subsets_phase2", "src/seq", "src/zs", "src/stats", "src/phase2")
    for p in sorted((REPO_DIR / folder).rglob("*"))
    if p.is_file() and "__pycache__" not in p.parts
]
frozen_paths += [
    REPO_DIR / rel
    for rel in ("configs/base.yaml", "configs/zs.yaml", "configs/seq.yaml", "configs/phase2.yaml", "src/common/constants.py")
]
frozen_snapshot = {p.relative_to(REPO_DIR).as_posix(): sha256_path(p) for p in frozen_paths}
frozen_file = ENV_DIR / "frozen_before.json"
if frozen_file.is_file():
    assert json.loads(frozen_file.read_text(encoding="utf-8")) == frozen_snapshot
else:
    frozen_file.write_text(json.dumps(frozen_snapshot, sort_keys=True, indent=2) + "\n", encoding="utf-8")

if DRIVE_SESSION_ROOT is not None:
    hydrate_llm_from_flat_drive(DRIVE_SESSION_ROOT, LLM_OUT_DIR, COMMIT, PROFILE_ID)

print("Commit:", COMMIT)
print("LLM Output dir:", LLM_OUT_DIR)

## 4. Kiểm tra `venv`/`ensurepip` & dọn virtualenv lỗi

In [ ]:
probe = subprocess.run([sys.executable, "-c", "import venv, ensurepip"], capture_output=True, text=True)
if probe.returncode != 0:
    py_mm = f"{sys.version_info.major}.{sys.version_info.minor}"
    apt_pkg = f"python{py_mm}-venv"
    print(f"Thiếu venv/ensurepip cho Python {py_mm}; đang cài gói hệ thống {apt_pkg}...")
    apt_upd = subprocess.run(["apt-get", "update", "-qq"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    apt_ins = subprocess.run(
        ["apt-get", "install", "-y", "-qq", apt_pkg], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    apt_log = (apt_upd.stdout or "") + "\n" + (apt_ins.stdout or "")
    (LOG_DIR / f"apt_venv_{SESSION_ID}.log").write_text(apt_log, encoding="utf-8")
    print(apt_log[-3000:])
    apt_upd.check_returncode()
    apt_ins.check_returncode()
    subprocess.run([sys.executable, "-c", "import venv, ensurepip"], check=True)

sys_py_ver = subprocess.check_output([sys.executable, "-c", "import platform; print(platform.python_version())"], text=True).strip()
print(f"Host sys.executable Python: {sys_py_ver} (profile expects venv Python: {PROFILES[PROFILE_ID]['python']})")

PYTHON = VENV_DIR / "bin" / "python"
PIP_BIN = VENV_DIR / "bin" / "pip"
if VENV_DIR.exists() and not (PYTHON.is_file() and PIP_BIN.is_file()):
    print("Phát hiện virtualenv tạo dở (thiếu python hoặc pip), đang xóa để tạo lại:", VENV_DIR)
    shutil.rmtree(VENV_DIR)
print("Venv support check: PASS")

## 5. Tạo virtualenv, cài package pin & cấu hình `RUN_ENV`

In [ ]:
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
assert PYTHON.is_file() and PIP_BIN.is_file()

pins_list = [f"{name}=={ver}" for name, ver in PROFILES[PROFILE_ID]["pins"].items()]
if PRECISION == "4bit":
    pins_list.append(PROFILES[PROFILE_ID]["bitsandbytes_spec"])

install = subprocess.run(
    [str(PYTHON), "-m", "pip", "install", *pins_list],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for p in (LOG_DIR / f"install_{SESSION_ID}.log", LOG_DIR / "install.log"):
    p.write_text(install.stdout, encoding="utf-8")
print(install.stdout[-4000:])
install.check_returncode()

freeze_out = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
for p in (ENV_DIR / f"pip-freeze_{SESSION_ID}.txt", ENV_DIR / "pip-freeze.txt"):
    p.write_text(freeze_out, encoding="utf-8")

pip_chk = subprocess.run([str(PYTHON), "-m", "pip", "check"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
for p in (ENV_DIR / f"pip_check_{SESSION_ID}.txt", ENV_DIR / "pip_check.txt"):
    p.write_text((pip_chk.stdout or "") + f"\n[pip_check_exit_code]={pip_chk.returncode}\n", encoding="utf-8")

RUN_ENV = os.environ.copy()
RUN_ENV.update(
    PYTHONUNBUFFERED="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
    USE_TF="0",
    USE_TORCH="1",
    TF_CPP_MIN_LOG_LEVEL="3",
)
RUN_ENV.pop("HF_HUB_OFFLINE", None)
RUN_ENV.pop("TRANSFORMERS_OFFLINE", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        RUN_ENV["HF_TOKEN"] = hf_token.strip()
        print("Loaded optional HF_TOKEN from Colab Secrets (redacted).")
except Exception:
    pass

def run_python(source: str) -> None:
    res = subprocess.run([str(PYTHON), "-u", "-c", source], cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(res.stdout, end="" if res.stdout.endswith("\n") else "\n")
    res.check_returncode()

print(f"Installed {len(pins_list)} pinned packages into {VENV_DIR} (pip check exit={pip_chk.returncode}).")

## 6. Preflight môi trường, xác minh revision SHA & giấy phép Hugging Face

In [ ]:
preflight_llm_script = r'''
import importlib.metadata as metadata
import json, platform, subprocess, sys
from pathlib import Path
import torch
from src.phase2.llm import load_llm_task_config, verify_hf_model_metadata
from src.phase2.run import atomic_json

ctx = json.loads(__CTX_JSON__)
profile_spec = ctx["profile_spec"]
env_dir = Path(ctx["env_dir"])

actual_pins = {}
mismatches = []
for pkg, exp_ver in profile_spec["pins"].items():
    try:
        got_ver = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        got_ver = None
    actual_pins[pkg] = got_ver
    if got_ver != exp_ver:
        mismatches.append(f"{pkg}: expected {exp_ver}, got {got_ver}")

py_ver = platform.python_version()
if py_ver != profile_spec["python"]:
    mismatches.append(f"python: expected {profile_spec['python']}, got {py_ver}")
if torch.__version__ != profile_spec["torch"]:
    mismatches.append(f"torch: expected {profile_spec['torch']}, got {torch.__version__}")
if torch.version.cuda != profile_spec["cuda"]:
    mismatches.append(f"cuda: expected {profile_spec['cuda']}, got {torch.version.cuda}")

hub_meta = verify_hf_model_metadata(ctx["model_name"], verify_license=True)
cfg_bundle = load_llm_task_config("configs/task11_llm.yaml")
assert int(cfg_bundle["llm"]["max_input_length"]) >= 2048

snapshot = {
    "profile_id": ctx["profile_id"],
    "session_id": ctx["session_id"],
    "commit": ctx["commit"],
    "python": py_ver,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_total_bytes": int(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else 0,
    "selected_precision": ctx["precision"],
    "selected_batch_size": ctx["batch_size"],
    "hub_meta": hub_meta,
    "pinned_packages_actual": actual_pins,
    "preflight_mismatches": mismatches,
    "preflight_status": "FAIL" if mismatches else "PASS",
}
atomic_json(env_dir / f"environment_{ctx['session_id']}.json", snapshot)
atomic_json(env_dir / "environment.json", snapshot)
if mismatches:
    raise AssertionError("Profile preflight FAILED: " + "; ".join(mismatches))
print("Verified HF model:", json.dumps(hub_meta, indent=2))
print("LLM environment & license preflight: PASS")
'''
run_python(preflight_llm_script.replace("__CTX_JSON__", repr(json.dumps({
    "profile_id": PROFILE_ID,
    "profile_spec": PROFILES[PROFILE_ID],
    "session_id": SESSION_ID,
    "commit": COMMIT,
    "env_dir": str(ENV_DIR),
    "model_name": MODEL_NAME,
    "precision": PRECISION,
    "batch_size": BATCH_SIZE,
}))))

## 7. Smoke test `dev-only` (`20` mẫu, kiểm tra chống cắt cụt câu hỏi)

In [ ]:
smoke_out = SMOKE_DIR / "llm"
smoke_cmd = [
    str(PYTHON), "-u", "-m", "src.phase2.llm", "dev-sweep",
    "--config", "configs/task11_llm.yaml",
    "--domain", "vsmec",
    "--model-name", MODEL_NAME,
    "--precision", PRECISION,
    "--batch-size", str(BATCH_SIZE),
    "--max-examples", "20",
    "--out-dir", str(smoke_out),
]
smoke_proc = subprocess.run(smoke_cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(LOG_DIR / f"smoke_{SESSION_ID}.log").write_text(smoke_proc.stdout, encoding="utf-8")
print(smoke_proc.stdout[-2500:])
smoke_proc.check_returncode()
assert not (smoke_out / "vsmec" / "test_summary_vsmec.csv").exists()
import pandas as pd
smoke_df = pd.read_csv(smoke_out / "vsmec" / "dev_sweep_vsmec.csv")
assert int(smoke_df["max_prompt_tokens"].max()) <= 2048
print(smoke_df[["strategy", "prompt_style", "dev_selection_score", "invalid_output_rate", "max_prompt_tokens"]])

if DRIVE_SESSION_ROOT is not None:
    for folder_name in ("environment", "logs"):
        for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                atomic_copy_file(item, DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name))
    print("Synced preflight env & smoke logs to Drive:", DRIVE_SESSION_ROOT)

## 8. Thực thi `dev-sweep` & `eval-test` (`vsmec`, `neuesc`) kèm đồng bộ Drive

In [ ]:
LLM_LOG_PATH = LOG_DIR / f"llm_run_{SESSION_ID}.log"

def sync_llm_domain_to_drive(domain: str) -> None:
    if DRIVE_SESSION_ROOT is None:
        return
    l_dom = LLM_OUT_DIR / domain
    d_dom = DRIVE_SESSION_ROOT / domain
    if not l_dom.is_dir():
        return
    for item in sorted(l_dom.rglob("*")):
        if item.is_file() and item.suffix != ".tmp":
            dst = d_dom / item.relative_to(l_dom)
            if dst.is_file() and dst.stat().st_size == item.stat().st_size and sha256_path(dst) == sha256_path(item):
                continue
            atomic_copy_file(item, dst)

def run_llm_subprocess(tag: str, domain: str, cmd: list[str]) -> None:
    with LLM_LOG_PATH.open("a", encoding="utf-8") as log_handle:
        start_offset = log_handle.tell()
        log_handle.write(f"\n=== [{tag} {domain}] start {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} ===\n")
        log_handle.flush()
        t0 = time.perf_counter()
        proc = subprocess.Popen(
            cmd, cwd=REPO_DIR, env=RUN_ENV, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True
        )
        last_hb = time.monotonic()
        try:
            while proc.poll() is None:
                time.sleep(3)
                if time.monotonic() - last_hb >= 30:
                    last_hb = time.monotonic()
                    sync_llm_domain_to_drive(domain)
                    tail = LLM_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-350:]
                    last_ln = [ln for ln in tail.splitlines() if ln.strip()][-1:] or ["..."]
                    print(f"  [{tag}/{domain} running {round(time.perf_counter() - t0)}s] {last_ln[0]}")
        except BaseException:
            if proc.poll() is None:
                os.killpg(proc.pid, signal.SIGINT)
                proc.wait(timeout=60)
            if DRIVE_SESSION_ROOT is not None and LLM_LOG_PATH.is_file():
                atomic_copy_file(LLM_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / LLM_LOG_PATH.name)
            raise

    with LLM_LOG_PATH.open("r", encoding="utf-8", errors="replace") as rh:
        rh.seek(start_offset)
        segment = rh.read()
    sync_llm_domain_to_drive(domain)
    if proc.returncode != 0:
        if DRIVE_SESSION_ROOT is not None and LLM_LOG_PATH.is_file():
            atomic_copy_file(LLM_LOG_PATH, DRIVE_SESSION_ROOT / "logs" / LLM_LOG_PATH.name)
        print(segment[-5000:])
        raise RuntimeError(f"[{tag}/{domain}] thất bại (exit={proc.returncode}).")
    print(f"[{tag}/{domain}] complete in {time.perf_counter() - t0:.1f}s")

for domain in ("vsmec", "neuesc"):
    run_llm_subprocess("DEV-SWEEP", domain, [
        str(PYTHON), "-u", "-m", "src.phase2.llm", "dev-sweep",
        "--config", "configs/task11_llm.yaml",
        "--domain", domain,
        "--model-name", MODEL_NAME,
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--seed", "0",
        "--out-dir", str(LLM_OUT_DIR),
    ])

for domain in ("vsmec", "neuesc"):
    sel_path = LLM_OUT_DIR / domain / f"selection_llm_{domain}.json"
    run_llm_subprocess("EVAL-TEST", domain, [
        str(PYTHON), "-u", "-m", "src.phase2.llm", "eval-test",
        "--config", "configs/task11_llm.yaml",
        "--domain", domain,
        "--model-name", MODEL_NAME,
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--selection-path", str(sel_path),
        "--out-dir", str(LLM_OUT_DIR),
    ])
print("Completed LLM dev-sweep and locked test evaluation for vsmec & neuesc.")

## 9. Kiểm tra `SKIP` & nghiệm thu độc lập toàn bộ artifact LLM

In [ ]:
verify_llm_script = r'''
import json
from pathlib import Path
import numpy as np
import pandas as pd
from src.common.hashing import sha256_file
from src.common.schema import ZS_PRED_COLUMNS
from src.phase2.llm import run_llm_dev_sweep, run_llm_test_eval

ctx = json.loads(__CTX_JSON__)
llm_dir = Path(ctx["llm_dir"])
env_dir = Path(ctx["env_dir"])
model_meta = {"model_name": ctx["model_name"], "precision": ctx["precision"]}

def must_not_run(_prompts):
    raise AssertionError("Predictor was called despite complete SKIP artifacts!")

for domain, expected_rows in (("vsmec", 692), ("neuesc", 6613)):
    dom_dir = llm_dir / domain
    sweep_df = pd.read_csv(dom_dir / f"dev_sweep_{domain}.csv")
    assert len(sweep_df) == 8, (domain, len(sweep_df))
    assert int(sweep_df["max_prompt_tokens"].max()) <= 2048

    shots_per_class = 1 if domain == "vsmec" else 2
    sw_skip = run_llm_dev_sweep(
        domain, must_not_run, model_meta=model_meta, shots_per_class=shots_per_class, seed=0, out_dir=llm_dir
    )
    assert len(sw_skip["skipped_tags"]) == 8, (domain, sw_skip["skipped_tags"])

    sel_path = dom_dir / f"selection_llm_{domain}.json"
    te_skip = run_llm_test_eval(
        domain, sel_path, must_not_run, model_meta=model_meta, out_dir=llm_dir
    )
    assert set(te_skip["skipped_modes"]) == {"zero_shot", "few_shot"}, (domain, te_skip["skipped_modes"])

    for mode_key in ("zero_shot", "few_shot"):
        tdir = dom_dir / f"test_{mode_key}"
        comp = json.loads((tdir / "completion.json").read_text(encoding="utf-8"))
        assert comp.get("complete") is True
        for fname, dig in comp["artifacts"].items():
            assert sha256_file(tdir / fname) == dig, (domain, mode_key, fname)
        meta = json.loads((tdir / "metadata.json").read_text(encoding="utf-8"))
        assert meta["pretraining_leakage_risk"] is True
        pred = pd.read_csv(tdir / "predictions_test.csv")
        assert tuple(pred.columns) == ZS_PRED_COLUMNS and len(pred) == expected_rows and pred["id"].is_unique
        p3 = pred[["p3_neg", "p3_neu", "p3_pos"]].to_numpy(float)
        assert np.isfinite(p3).all() and (p3 >= 0).all() and np.allclose(p3.sum(axis=1), 1.0, atol=1e-6)
        if domain == "vsmec":
            assert int(pred["in_strict"].astype(bool).sum()) == 526

frozen = json.loads((env_dir / "frozen_before.json").read_text(encoding="utf-8"))
assert all(sha256_file(Path(rel)) == dig for rel, dig in frozen.items()), "Tài nguyên đóng băng bị thay đổi!"
print("Independent verification of LLM dev-sweep, eval-test, SKIP & frozen files: PASS")
'''
run_python(verify_llm_script.replace("__CTX_JSON__", repr(json.dumps({
    "llm_dir": str(LLM_OUT_DIR), "env_dir": str(ENV_DIR), "model_name": MODEL_NAME, "precision": PRECISION
}))))

## 10. Benchmark dual-mode (`zero_shot` & `few_shot`), xuất bảng tổng hợp & đồng bộ Drive

In [ ]:
for domain in ("vsmec", "neuesc"):
    sel_path = LLM_OUT_DIR / domain / f"selection_llm_{domain}.json"
    b_res = subprocess.run([
        str(PYTHON), "-u", "-m", "src.phase2.llm", "benchmark",
        "--config", "configs/task11_llm.yaml",
        "--domain", domain,
        "--model-name", MODEL_NAME,
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--selection-path", str(sel_path),
        "--out-dir", str(LLM_OUT_DIR),
    ], cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(f"=== Benchmark ({domain}) ===")
    print(b_res.stdout)
    b_res.check_returncode()

sum_cmd = [
    str(PYTHON), "-u", "-m", "src.phase2.task11", "summarize",
    "--config", "configs/phase2.yaml",
    "--llm-results-dir", str(LLM_OUT_DIR),
    "--llm-cost-dir", str(LLM_OUT_DIR),
    "--out-dir", str(RESULTS_DIR / "tables"),
    "--report-path", str(OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md"),
]
if MOUNT_DRIVE and DRIVE_TASK08_RESULTS_DIR.strip() and Path(DRIVE_TASK08_RESULTS_DIR.strip()).is_dir():
    sum_cmd += ["--visobert-results-dir", DRIVE_TASK08_RESULTS_DIR.strip()]
if MOUNT_DRIVE and DRIVE_TASK11_XLMR_DIR.strip() and Path(DRIVE_TASK11_XLMR_DIR.strip()).is_dir():
    x_dir = Path(DRIVE_TASK11_XLMR_DIR.strip())
    sum_cmd += ["--xlmr-results-dir", str(x_dir)]
    if (x_dir / "xlmr_zs").is_dir():
        sum_cmd += ["--xlmr-zs-dir", str(x_dir / "xlmr_zs")]
    if (x_dir / "cost_encoder_inference.json").is_file():
        sum_cmd += ["--encoder-cost-path", str(x_dir / "cost_encoder_inference.json")]

sum_res = subprocess.run(sum_cmd, cwd=REPO_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(sum_res.stdout)
sum_res.check_returncode()
print((OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md").read_text(encoding="utf-8"))

if DRIVE_SESSION_ROOT is not None:
    for item in sorted(LLM_OUT_DIR.rglob("*")):
        if item.is_file() and item.suffix != ".tmp":
            dst = DRIVE_SESSION_ROOT / item.relative_to(LLM_OUT_DIR)
            atomic_copy_file(item, dst)
            assert sha256_path(dst) == sha256_path(item)
    for tbl in sorted((RESULTS_DIR / "tables").glob("*.csv")):
        dst = DRIVE_SESSION_ROOT / "tables" / tbl.name
        atomic_copy_file(tbl, dst)
        assert sha256_path(dst) == sha256_path(tbl)
    rep_src = OUTPUT_DIR / "reports" / "phase2" / "task11_xlmr_llm_10pct.md"
    rep_dst = DRIVE_SESSION_ROOT / "task11_xlmr_llm_10pct.md"
    atomic_copy_file(rep_src, rep_dst)
    assert sha256_path(rep_dst) == sha256_path(rep_src)
    for folder_name in ("environment", "logs"):
        for item in sorted((OUTPUT_DIR / folder_name).rglob("*")):
            if item.is_file() and item.suffix != ".tmp":
                dst = DRIVE_SESSION_ROOT / folder_name / item.relative_to(OUTPUT_DIR / folder_name)
                atomic_copy_file(item, dst)
                assert sha256_path(dst) == sha256_path(item)
    print("Synced & SHA-256 verified all Task 11 LLM deliverables on flat Drive:", DRIVE_SESSION_ROOT)